In [1]:
import re
import pandas as pd

import nltk
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [2]:
df = pd.read_csv("text_data.csv")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return " ".join(tokens)

df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text", "label"]].head()


,text,clean_text,label
0,thirtysomething scientists unveil doomsday clo...,thirtysomething scientist unveil doomsday cloc...,1
1,dem rep. totally nails why congress is falling...,dem rep totally nail congress falling short ge...,0
2,eat your veggies: 9 deliciously different recipes,eat veggie deliciously different recipe,0
3,inclement weather prevents liar from getting t...,inclement weather prevents liar getting work,1
4,mother comes pretty close to using word 'strea...,mother come pretty close using word streaming ...,1


In [3]:
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [4]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("model", LogisticRegression(max_iter=5000))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


In [5]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")
print(f"F1 score:  {f1:.3f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy:  0.779
Precision: 0.793
Recall:    0.727
F1 score:  0.758
Confusion matrix:
 [[2479  518]
 [ 745 1982]]


In [6]:
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="f1")
print("CV F1:", cv_scores.mean())


CV F1: 0.7581658884622786


In [7]:
param_grid = {
    "tfidf__max_features": [3000, 5000, 8000],
    "model__C": [0.1, 1, 10]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="f1")
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV F1:", grid_search.best_score_)


Best params: {'model__C': 10, 'tfidf__max_features': 8000}
Best CV F1: 0.7683188003863661


In [8]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("Baseline -> F1: {:.3f}, Accuracy: {:.3f}".format(f1, acc))
print("Tuned    -> F1: {:.3f}, Accuracy: {:.3f}".format(
    f1_score(y_test, y_pred_best),
    accuracy_score(y_test, y_pred_best)
))


Baseline -> F1: 0.758, Accuracy: 0.779
Tuned    -> F1: 0.775, Accuracy: 0.789
